# Silver: Discount — Bronze → Silver Cleansing

Operations performed:
- Deduplicate
- Trim / case normalisation
- Type casting
- Null handling
- Incremental load via `ingest_ts` watermark

In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")


In [0]:
silverTablName = f"saleslake_{env}.silver_{env}.cleanedDiscount"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawDiscount"
print(f"Source : {bronzeTablName}")
print(f"Target : {silverTablName}")


In [0]:
spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(discount_id) AS INT)            AS discount_id,
    UPPER(TRIM(discount_code))                AS discount_code,
    TRIM(discount_name)                       AS discount_name,
    UPPER(TRIM(discount_type))                AS discount_type,
    CAST(TRIM(discount_value) AS DECIMAL(10,2)) AS discount_value,
    CAST(TRIM(min_purchase_amount) AS DECIMAL(18,2)) AS min_purchase_amount,
    CAST(TRIM(max_discount_amount) AS DECIMAL(18,2)) AS max_discount_amount,
    TO_DATE(TRIM(valid_from), 'yyyy-MM-dd')   AS valid_from,
    TO_DATE(TRIM(valid_to),   'yyyy-MM-dd')   AS valid_to,
    UPPER(TRIM(applicable_segment))           AS applicable_segment,
    UPPER(TRIM(applicable_category))          AS applicable_category,
    CAST(TRIM(usage_limit_per_customer) AS INT) AS usage_limit_per_customer,
    CAST(TRIM(total_usage_limit) AS INT)      AS total_usage_limit,
    UPPER(TRIM(is_active))                    AS is_active,
    TO_DATE(TRIM(created_date), 'yyyy-MM-dd') AS created_date,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTablName}
)
ORDER BY discount_id
""")

print(f"Silver load complete for {silverTablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()
